In [ ]:
import sys
import traceback
import logging
import portalocker
import os
import re

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

# 경로 설정 - 플랫폼에 따라 다르게 처리
if os.name == 'nt':  # Windows
    sys.path.insert(0, r"Y:/git/pyaedt_library/src/")
else:  # Linux/Unix
    # Linux 서버 경로들 시도
    possible_paths = [
        # r"/gpfs/home1/r1jae262/jupyter/git/pyaedt_library/src/",
        r"../pyaedt_library/src/",
        os.path.abspath(os.path.join(BASE_DIR, "../git/pyaedt_library/src/")),
    ]
    for path in possible_paths:
        if os.path.exists(path):
            sys.path.insert(0, path)
            break

import pyaedt_module
from pyaedt_module.core import pyDesktop
import os
import time
from datetime import datetime

import math
import copy

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


import platform
import csv



from module.input_parameter import create_input_parameter, set_design_variables, validation_check
from module.modeling import create_core, create_coil, create_coil_section




class Simulation() :

    def __init__(self, desktop=None) :

        self.NUM_CORE = 4
        self.NUM_TASK = 1
        self.desktop = desktop


    def create_simulation_name(self):

        file_path = "./simulation_num.txt"

        # 파일이 존재하지 않으면 생성
        if not os.path.exists(file_path):
            with open(file_path, "w", encoding="utf-8") as file:
                file.write("1")

        # 읽기/쓰기 모드로 파일 열기
        with open(file_path, "r+", encoding="utf-8") as file:
            # 파일 잠금: LOCK_EX는 배타적 잠금,  블로킹 모드로 실행 (Windows/Linux 호환)
            portalocker.lock(file, portalocker.LOCK_EX)

            # 파일에서 값 읽기
            content = int(file.read().strip())
            self.num = content
            self.PROJECT_NAME = f"simulation{content}"
            content += 1

            # 파일 포인터를 처음으로 되돌리고, 파일 내용 초기화 후 새 값 쓰기
            file.seek(0)
            file.truncate()
            file.write(str(content))




def run(simulation=None):



    sim = simulation

    sim.create_simulation_name()

    # simulation 디렉토리 생성 (존재하지 않으면)
    simulation_dir = "./simulation"
    if not os.path.exists(simulation_dir):
        os.makedirs(simulation_dir, exist_ok=True)
    
    # 절대 경로로 변환
    project_path = os.path.abspath(os.path.join(simulation_dir, sim.PROJECT_NAME))
    
    # desktop이 None이거나 유효하지 않은지 확인
    if sim.desktop is None:
        raise RuntimeError("Desktop instance is None. Cannot create project.")
    
    try:
        project1 = sim.desktop.create_project(path=project_path, name=sim.PROJECT_NAME)
    except Exception as e:
        error_msg = f"Failed to create project '{sim.PROJECT_NAME}' at path '{project_path}': {e}\n"
        print(error_msg, file=sys.stderr)
        sys.stderr.flush()
        raise
    
    
    design1 = project1.create_design(name="maxwell_design", solver="maxwell3d", solution="AC Magnetic")

    # input_parameter = sim1.create_input_parameter()
    # sim1.set_variable(input_parameter)

    sim.project = project1
    sim.design1 = design1



itr = 0
GUI = False


# with pyDesktop(version=None, non_graphical=GUI, close_on_exit=False, new_desktop=True) as desktop:

desktop = pyDesktop(version=None, non_graphical=GUI, close_on_exit=False, new_desktop=True)

sim = Simulation(desktop=desktop)
# run(simulation=sim)
sim.create_simulation_name()

# simulation 디렉토리 생성 (존재하지 않으면)
simulation_dir = "./simulation"
if not os.path.exists(simulation_dir):
    os.makedirs(simulation_dir, exist_ok=True)

# 절대 경로로 변환
project_path = os.path.abspath(os.path.join(simulation_dir, sim.PROJECT_NAME))

# desktop이 None이거나 유효하지 않은지 확인
if sim.desktop is None:
    raise RuntimeError("Desktop instance is None. Cannot create project.")

try:
    project1 = sim.desktop.create_project(path=project_path, name=sim.PROJECT_NAME)
except Exception as e:
    error_msg = f"Failed to create project '{sim.PROJECT_NAME}' at path '{project_path}': {e}\n"
    print(error_msg, file=sys.stderr)
    sys.stderr.flush()
    raise


design1 = project1.create_design(name="maxwell_design", solver="maxwell3d", solution="AC Magnetic")

# input_parameter = sim1.create_input_parameter()
# sim1.set_variable(input_parameter)

sim.project = project1
sim.design1 = design1

oDesign = sim.design1.odesign
oDesign.SetDesignSettings(
	[
		"NAME:Design Settings Data",
		"Allow Material Override:=", False,
		"Perform Minimal validation:=", False,
		"EnabledObjects:="	, [],
		"PerfectConductorThreshold:=", 1E+30,
		"InsulatorThreshold:="	, 1,
		"SolveFraction:="	, False,
		"Multiplier:="		, "1",
		"SkipMeshChecks:="	, True
	], 
	[
		"NAME:Model Validation Settings",
		"EntityCheckLevel:="	, "Strict",
		"IgnoreUnclassifiedObjects:=", False,
		"SkipIntersectionChecks:=", False
	])



while True :
    sim.input_df = create_input_parameter()
    result, df_plus = validation_check(sim.input_df)
    if result :
        break

set_design_variables(sim.design1, sim.input_df)

sim.design1.set_power_ferrite(cm=1.38, x=1.51, y=1.74) # 1K101 parameter [W/m^3]
power_ferrite_mat = sim.design1.materials["power_ferrite"]
power_ferrite_mat.permeability = "3000"

sim.design1.core = create_core(design=sim.design1, name="core", core_material="power_ferrite")




# Tx winding 생성
Tx_windings, Tx_N, Tx_coil_width, Tx_coil_height, Tx_coil_gap_x, Tx_coil_gap_z = create_coil(
    design = sim.design1,
    name = "Tx",
    window_height = df_plus["nwh1"].iloc[0],
    window_length = df_plus["nwl1"].iloc[0],
    window_layer = 1,
    N_input = df_plus["N1"].iloc[0],
    width_fill_factor = df_plus["wff1"].iloc[0],
    space_length = df_plus["sl1"].iloc[0],
    space_width = df_plus["sw1"].iloc[0],
    shape = "circle",
    offset = [0,0,0],
    color = [255,10,10]
)

l1 = df_plus["l1"].iloc[0]
l2 = df_plus["l2"].iloc[0]




Rx_windings1, Rx_N1, Rx_coil_width1, Rx_coil_height1, Rx_coil_gap_x1, Rx_coil_gap_z1 = create_coil(
    design = sim.design1,
    name = "Rx_center",
    window_height = df_plus["nwh2"].iloc[0],
    window_length = df_plus["nwl2_main"].iloc[0],
    window_layer = df_plus["N2_main"].iloc[0],
    N_input = 1,
    width_fill_factor = df_plus["wff2"].iloc[0],
    space_length = df_plus["sl2_main"].iloc[0],
    space_width = df_plus["sw2_main"].iloc[0],
    shape = "rectangle",
    offset = [0,0,0],
    color = [10, 10, 255]
)   

Rx_windings2, Rx_N2, Rx_coil_width2, Rx_coil_height2, Rx_coil_gap_x2, Rx_coil_gap_z2 = create_coil(
    design = sim.design1,
    name = "Rx_side1",
    window_height = df_plus["nwh2"].iloc[0],
    window_length = df_plus["nwl2_side"].iloc[0],
    window_layer = df_plus["N2_side"].iloc[0],
    N_input = 1,
    width_fill_factor = df_plus["wff2"].iloc[0],
    space_length = df_plus["sl2_side"].iloc[0],
    space_width = df_plus["sw2_side"].iloc[0],
    shape = "rectangle",
    offset=[(-l1-l2-l1/2),0,0],
    color = [10, 10, 255]
)   

Rx_windings3, Rx_N3, Rx_coil_width3, Rx_coil_height3, Rx_coil_gap_x3, Rx_coil_gap_z3 = create_coil(
    design = sim.design1,
    name = "Rx_side2",
    window_height = df_plus["nwh2"].iloc[0],
    window_length = df_plus["nwl2_side"].iloc[0],
    window_layer = df_plus["N2_side"].iloc[0],
    N_input = 1,
    width_fill_factor = df_plus["wff2"].iloc[0],
    space_length = df_plus["sl2_side"].iloc[0],
    space_width = df_plus["sw2_side"].iloc[0],
    shape = "rectangle",
    offset = [(l1+l2+l1/2),0,0],
    color = [10, 10, 255]
)   


PyAEDT INFO: Python version 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\Public\Documents\ESTsoft\CreatorTemp\pyaedt_peets_e2d3300b-e480-44d9-98c6-aac82ebf6937.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 57216.
PyAEDT INFO: Electronics Desktop started on gRPC port: 57216 after 47.6662757396698 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v252\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.2 version started with process ID 329968.
PyAEDT INFO: Python version 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO

In [12]:
nwl1 = float(df_plus["nwl1"])
nwl2_main = float(df_plus["nwl2_main"])
nwl2_side = float(df_plus["nwl2_side"])
nwh1 = float(df_plus["nwh1"])
nwh2 = float(df_plus["nwh2"])

sl1 = float(df_plus["sl1"])
sw1 = float(df_plus["sw1"])
sl2_main = float(df_plus["sl2_main"])
sw2_main = float(df_plus["sw2_main"])
sl2_side = float(df_plus["sl2_side"])
sw2_side = float(df_plus["sw2_side"])




In [25]:

box_center = design1.modeler.create_box(origin=[-(sl2_main+2*nwl2_main)/2, -(sw2_main+2*nwl2_main)/2, -(nwh2)/2], sizes=[sl2_main+2*nwl2_main, sw2_main+2*nwl2_main, nwh2], name="box_center")
box_center_sub = design1.modeler.create_box(origin=[-(sl2_main)/2, -(sw2_main)/2, -(nwh2)/2], sizes=[sl2_main, sw2_main, nwh2], name="box_center_sub")
design1.modeler.subtract(blank_list=[box_center], tool_list=[box_center_sub], keep_originals=False)

box_side1 = design1.modeler.create_box(origin=[-(sl2_side+2*nwl2_side)/2+(l1+l2+l1/2), -(sw2_side+2*nwl2_side)/2, -(nwh2)/2], sizes=[sl2_side+2*nwl2_side, sw2_side+2*nwl2_side, nwh2], name="box_side1")
box_side1_sub = design1.modeler.create_box(origin=[-(sl2_side)/2+(l1+l2+l1/2), -(sw2_side)/2, -(nwh2)/2], sizes=[sl2_side, sw2_side, nwh2], name="box_side1_sub")
design1.modeler.subtract(blank_list=[box_side1], tool_list=[box_side1_sub], keep_originals=False)

box_side2 = design1.modeler.create_box(origin=[-(sl2_side+2*nwl2_side)/2-(l1+l2+l1/2), -(sw2_side+2*nwl2_side)/2, -(nwh2)/2], sizes=[sl2_side+2*nwl2_side, sw2_side+2*nwl2_side, nwh2], name="box_side2")
box_side2_sub = design1.modeler.create_box(origin=[-(sl2_side)/2-(l1+l2+l1/2), -(sw2_side)/2, -(nwh2)/2], sizes=[sl2_side, sw2_side, nwh2], name="box_side2_sub")
design1.modeler.subtract(blank_list=[box_side2], tool_list=[box_side2_sub], keep_originals=False)

design1.modeler.subtract(blank_list=[box_center], tool_list=Rx_windings1, keep_originals=True)
design1.modeler.subtract(blank_list=[box_side1], tool_list=Rx_windings2, keep_originals=True)
design1.modeler.subtract(blank_list=[box_side2], tool_list=Rx_windings3, keep_originals=True)


PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec


True

In [2]:
Tx_neg_sheets, Tx_pos_sheets = create_coil_section(design=sim.design1, winding_obj=Tx_windings, sheet_prefix = None, plane = "ZX", rename_faces = False)

Rx_neg_sheets_center, Rx_pos_sheets_center = create_coil_section(design=sim.design1, winding_obj=Rx_windings1, sheet_prefix = None, plane = "ZX", rename_faces = False)
if df_plus["N2_side"].iloc[0] != 0 :
    Rx_neg_sheets_side1, Rx_pos_sheets_side1 = create_coil_section(design=sim.design1, winding_obj=Rx_windings2, sheet_prefix = None, plane = "ZX", rename_faces = False)
    Rx_neg_sheets_side2, Rx_pos_sheets_side2 = create_coil_section(design=sim.design1, winding_obj=Rx_windings3, sheet_prefix = None, plane = "ZX", rename_faces = False)

In [3]:
tx_winding = sim.design1.assign_winding(
    assignment=[], 
    winding_type="Current", 
    is_solid=True, 
    current=f"{1000*math.sqrt(2)}A",
    name="Tx_winding"
)
rx_winding1 = sim.design1.assign_winding(
    assignment=[], 
    winding_type="Current", 
    is_solid=True, 
    current=f"{100*math.sqrt(2)}A",
    name="Rx_winding"
)

PyAEDT INFO: Boundary Winding Tx_winding has been created.
PyAEDT INFO: Boundary Winding Rx_winding has been created.


In [4]:
Tx_coil = []
Rx_coil = []

import re

for idx, sheet in enumerate(Tx_neg_sheets, start=1):
    coil = sim.design1.assign_coil(sheet, conductors_number=1, polarity="Positive", name=f"Tx_coil{idx}")
    Tx_coil.append(coil)

for idx, sheet in enumerate(Rx_neg_sheets_center, start=1):
    coil = sim.design1.assign_coil(sheet, conductors_number=1, polarity="Negative", name=f"Rx_center_coil{idx}")
    Rx_coil.append(coil)

if df_plus["N2_side"].iloc[0] != 0 :
    for idx, sheet in enumerate(Rx_neg_sheets_side1 + Rx_neg_sheets_side2, start=1):
        coil = sim.design1.assign_coil(sheet, conductors_number=1, polarity="Positive", name=f"Rx_side_coil{idx}")
        Rx_coil.append(coil)





sim.design1.add_winding_coils(assignment="Tx_winding", coils=[coil.name for coil in Tx_coil])
sim.design1.add_winding_coils(assignment="Rx_winding", coils=[coil.name for coil in Rx_coil])

sim.design1.assign_matrix(matrix_name="Matrix", assignment=["Tx_winding", "Rx_winding"])

PyAEDT INFO: Boundary CoilTerminal Tx_coil1 has been created.
PyAEDT INFO: Boundary CoilTerminal Tx_coil2 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil1 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil2 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil3 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil4 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil5 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil6 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil7 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil8 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil9 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil10 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil11 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_coil12 has been created.
PyAEDT INFO: Boundary CoilTerminal Rx_center_

In [5]:
freq = 1e+3

mu0 = 4 * math.pi * 1e-7
mu_copper = mu0 
sigma_copper = 58000000
omega = 2 * math.pi * freq
skin_depth = math.sqrt(2 / (omega * mu_copper * sigma_copper)) * 1e3 # in mm


Tx_skin_depth_mesh = sim.design1.mesh.assign_skin_depth(
    assignment=Tx_windings,
    skin_depth=f'{skin_depth}mm',
    triangulation_max_length='50mm',
    layers_number="2",
    name="Tx_winding_skin_depth"
)

PyAEDT INFO: Mesh class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Mesh class has been initialized! Elapsed time: 0m 0sec


In [7]:
sim.air_region = sim.design1.modeler.create_air_region(x_pos=100.0, y_pos=100.0, z_pos=100.0, x_neg=100.0, y_neg=100.0, z_neg=100.0, is_percentage=True)
sim.design1.assign_radiation(assignment=[sim.air_region.name], radiation="Radiation")

PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Boundary Radiation Radiation has been created.


In [6]:



sim.design1.setup = sim.design1.create_setup(name = "Setup1")
sim.design1.setup.properties["Max. Number of Passes"] = 6 # 10
sim.design1.setup.properties["Min. Number of Passes"] = 1
sim.design1.setup.properties["Min. Converged Passes"] = 1
sim.design1.setup.properties["Percent Error"] = 2.5 # 2.5
sim.design1.setup.properties["Frequency Setup"] = f"1kHz"




In [7]:
sim.design1.setup.analyze(cores=4)

PyAEDT INFO: Project simulation98 Saved correctly
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 3D correctly changed.
PyAEDT INFO: Solving design setup Setup1
PyAEDT INFO: Design setup Setup1 solved correctly in 0.0h 12.0m 6.0s
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 3D correctly changed.


In [8]:
params = [
    ["Matrix.L(Tx_winding,Tx_winding)", f"Ltx", "uH"],
    ["Matrix.L(Rx_winding,Rx_winding)", f"Lrx", "uH"],
    ["Matrix.L(Tx_winding,Rx_winding)", f"M", "uH"],
    ["abs(Matrix.CplCoef(Tx_winding,Rx_winding))", f"k", ""],
    ["Matrix.L(Tx_winding,Tx_winding)*(abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Lmt", "uH"],
    ["Matrix.L(Rx_winding,Rx_winding)*(abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Lmr", "uH"],
    ["Matrix.L(Tx_winding,Tx_winding)*(1-abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Llt", "uH"],
    ["Matrix.L(Rx_winding,Rx_winding)*(1-abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Llr", "uH"],
    ["PerWindingSolidLoss(Tx_winding)", f"Tx_loss", "W"],
    ["PerWindingSolidLoss(Rx_winding)", f"Rx_loss", "W"],
]

dir = sim.project.path
mod = "write"
import_report = None
report_name = "magnetic_report"
file_name = "magnetic_report"

report1, df1 = sim.design1.get_magnetic_parameter(dir=dir, parameters=params, mod=mod, import_report=import_report, report_name=report_name, file_name=file_name)

PyAEDT INFO: Parsing y:\git\MFT_1MW_2026\simulation\simulation106\simulation106.aedt.
PyAEDT INFO: File y:\git\MFT_1MW_2026\simulation\simulation106\simulation106.aedt correctly loaded. Elapsed time: 0m 0sec
PyAEDT INFO: aedt file load time 0.40115857124328613
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec
PyAEDT WARNING: No report category provided. Automatically identified AC Magnetic


In [9]:
df_plus

,N1,N2,N2_main,N2_side,w1,l1,l2,h1,w1c_space_x,w1w2_space_x,w2c_space_x,w1c_space_y,w1w2_space_y,w2w2_space_y,w2c_space_y,w1c_space_z,w2c_space_z,window_ratio,wh1,wh2,wff1,wff2,nwl_t,nwl1,nwl2,nwl2_main,nwl2_side,nwh1,nwh2,coil_width1,coil_width2,coil_gap_layer,coil_gap_height,fill_factor,sl1,sw1,sl2_main,sw2_main,sl2_side,sw2_side
0,3,30,18,12,335,99,302.5,713,32.1,46.3,42.2,48.1,18.1,48.6,46.4,24.8,44.9,0.33,0.81,0.9,1.0,0.56,141.3,46.629,94.671,56.8026,37.8684,537.354,560.88,46.629,1.767192,1.436388,198.7335,0.260326,294.2,399.2,423.658,585.058,191.8,419.4


In [10]:
time.sleep(5)

oProject = sim.project
oProject.CopyDesign(sim.design1.name)
oProject.Paste()

sim.design2 = sim.design1.get_active_design()

sim.design2.delete_mesh(Tx_skin_depth_mesh)

V = 1000
Im = V / 2 / 3.141592 / 1e+3 / float(df1["Lmt"]*1e-6)

sim.design2.Tx_winding, sim.design2.Rx_winding = sim.design2.get_excitation(excitation_name=["Tx_winding", "Rx_winding"])

sim.design2.Tx_winding["Current"] = f'{Im} * sqrt(2)A'
sim.design2.Rx_winding["Current"] = '0 * sqrt(2)A'

sim.design2.core = sim.design2.model3d.find_object(sim.design1.core)

sim.design2.set_core_losses(assignment=sim.design2.core, core_loss_on_field=True)



PyAEDT INFO: Active Design set to maxwell_design1
PyAEDT INFO: Active Design set to maxwell_design1
PyAEDT INFO: Active Design set to maxwell_design1
PyAEDT INFO: Python version 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Returning found Desktop session with PID 296608!
PyAEDT INFO: Project simulation98 set to active.
PyAEDT INFO: Active Design set to maxwell_design1
PyAEDT INFO: Aedt Objects correctly read
PyAEDT WARNING: Key Current not found. Trying to applying new key 
PyAEDT WARNING: Key Current not found. Trying to applying new key 
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec


True

In [11]:
sim.design2.setup = sim.design2.get_setup(name=sim.design1.setup.name)

sim.design2.setup.analyze(cores=4)

PyAEDT INFO: Project simulation98 Saved correctly
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 3D correctly changed.
PyAEDT INFO: Solving design setup Setup1
PyAEDT INFO: Design setup Setup1 solved correctly in 0.0h 6.0m 21.0s
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 3D correctly changed.


In [12]:
params = [
    ["Matrix.L(Tx_winding,Tx_winding)", f"Ltx", "uH"],
    ["Matrix.L(Rx_winding,Rx_winding)", f"Lrx", "uH"],
    ["Matrix.L(Tx_winding,Rx_winding)", f"M", "uH"],
    ["abs(Matrix.CplCoef(Tx_winding,Rx_winding))", f"k", ""],
    ["Matrix.L(Tx_winding,Tx_winding)*(abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Lmt", "uH"],
    ["Matrix.L(Rx_winding,Rx_winding)*(abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Lmr", "uH"],
    ["Matrix.L(Tx_winding,Tx_winding)*(1-abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Llt", "uH"],
    ["Matrix.L(Rx_winding,Rx_winding)*(1-abs(Matrix.CplCoef(Tx_winding,Rx_winding))^2)", f"Llr", "uH"],
    ["CoreLoss", f"core_loss", "W"],
]

dir = sim.project.path
mod = "write"
import_report = None
report_name = "magnetic_report2"
file_name = "magnetic_report2"

report2, df2 = sim.design2.get_magnetic_parameter(dir=dir, parameters=params, mod=mod, import_report=import_report, report_name=report_name, file_name=file_name)

PyAEDT INFO: Parsing y:\git\MFT_1MW_2026\simulation\simulation98\simulation98.aedt.
PyAEDT INFO: File y:\git\MFT_1MW_2026\simulation\simulation98\simulation98.aedt correctly loaded. Elapsed time: 0m 0sec
PyAEDT INFO: aedt file load time 0.38872194290161133
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec
PyAEDT WARNING: No report category provided. Automatically identified AC Magnetic


In [13]:
result = pd.concat([df_plus, df1, df2], axis=1)
result

,N1,N2,N2_main,N2_side,w1,l1,l2,h1,w1c_space_x,w1w2_space_x,w2c_space_x,w1c_space_y,w1w2_space_y,w2w2_space_y,w2c_space_y,w1c_space_z,w2c_space_z,window_ratio,wh1,wh2,wff1,wff2,nwl_t,nwl1,nwl2,nwl2_main,nwl2_side,nwh1,nwh2,coil_width1,coil_width2,coil_gap_layer,coil_gap_height,fill_factor,sl1,sw1,sl2_main,sw2_main,sl2_side,sw2_side,Ltx,Lrx,M,k,Lmt,Lmr,Llt,Llr,Tx_loss,Rx_loss,Ltx,Lrx,M,k,Lmt,Lmr,Llt,Llr,core_loss
0,3,30,18,12,335,99,302.5,713,32.1,46.3,42.2,48.1,18.1,48.6,46.4,24.8,44.9,0.33,0.81,0.9,1.0,0.56,141.3,46.629,94.671,56.8026,37.8684,537.354,560.88,46.629,1.767192,1.436388,198.7335,0.260326,294.2,399.2,423.658,585.058,191.8,419.4,1010.433845,100554.542613,10049.720423,0.997009,1004.398986,99953.976299,6.034859,600.566314,364.795595,91.978853,1015.197991,101007.732032,10094.398887,0.996845,1008.802859,100371.444587,6.395132,636.287445,8525.72


In [14]:
freq = 1e+3
B = 0.1

cm=1.38
x=1.51
y=1.74

P = cm * freq**x + B**y

P/1000

# P_W_kg = P / 7.18 * 1e3  # W/m^3 to W/kg (1 W/kg = 7.18 mW/cm^3)
# P_W_kg * 1500

46.760511744218526